# Silver: Change Data Capture Detection

Detects changes (INSERT, UPDATE, DELETE) between staging and current silver tables.

## Logic

1. **INSERT**: New jobs in staging not in current
2. **UPDATE**: Jobs in both with different record_hash
3. **DELETE** (source-aware with staleness window):
   - Only checks jobs from the SAME source as the current batch
   - Only soft-deletes jobs not seen in the last N days (default: 7)
   - Prevents false deletions from batch processing order or multi-source data

## Architecture

**Input**: `silver.silver_jobs_staging`  
**Output**: `silver.silver_jobs_current`  
**Mode**: Incremental (process new batches only)

## Batch Processing

- Tracks processed batches in `metadata.staging_to_current_batches`
- Automatically processes all unprocessed staging batches
- Idempotent: safe to re-run

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for all unprocessed)")
dbutils.widgets.dropdown("enable_deletes", "true", ["true", "false"], "Enable soft deletes")
dbutils.widgets.text("staleness_days", "7", "Days before marking unseen jobs as deleted")

batch_id = dbutils.widgets.get("batch_id").strip()
enable_deletes = dbutils.widgets.get("enable_deletes") == "true"
staleness_days = int(dbutils.widgets.get("staleness_days"))

In [0]:
import json
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

In [0]:
# Create metadata table to track CDC-processed batches
metadata_table = f"{METADATA_SCHEMA}.staging_to_current_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  inserts INT,
  updates INT,
  deletes INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING,
  PRIMARY KEY (batch_id, source_name)
)
USING DELTA
COMMENT 'Tracks which staging batches have been processed through CDC to current'
""")

# Define metadata schema for recording batch processing
metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("inserts", IntegerType(), True),
    StructField("updates", IntegerType(), True),
    StructField("deletes", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# Create current table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_jobs_current (
  enterprise_job_id STRING NOT NULL,
  source_name STRING,
  source_job_id STRING,
  source_job_key STRING,
  company_name_raw STRING,
  company_name_norm STRING,
  title_raw STRING,
  title_normalized STRING,
  description_raw STRING,
  location_raw STRING,
  location_norm STRING,
  remote_type STRING,
  source_url STRING,
  posted_at TIMESTAMP,
  last_seen TIMESTAMP,
  is_active BOOLEAN,
  soft_delete_flag BOOLEAN,
  soft_delete_reason STRING,
  record_hash STRING,
  current_batch_id STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
""")

# Identify unprocessed batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
else:
    # Find all batches in staging
    all_batches_query = f"""
        SELECT DISTINCT batch_id, source_name
        FROM {SILVER_SCHEMA}.silver_jobs_staging
    """
    
    all_batches_df = spark.sql(all_batches_query)
    
    # Find already processed batches
    processed_batches_df = spark.sql(f"""
        SELECT DISTINCT batch_id, source_name
        FROM {metadata_table}
        WHERE status = 'success'
    """)
    
    # Find unprocessed batches (in staging but not in metadata)
    unprocessed_df = all_batches_df.join(
        processed_batches_df,
        on=["batch_id", "source_name"],
        how="left_anti"
    ).orderBy("batch_id", "source_name")
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unprocessed_df.limit(10).collect()]
    
    num_processed_batches = processed_batches_df.count()
    print(f"Number of batches already in the table: {num_processed_batches}")
    
    if not batches_to_process:
        dbutils.notebook.exit({"status": "success", "message": "No unprocessed batches", "total_changes": 0, "processed_batches": num_processed_batches})
    
    print(f"Found {len(batches_to_process)} unprocessed batch(es) to process")

In [0]:
from delta.tables import DeltaTable

# Initialize tracking
total_inserts = 0
total_updates = 0
total_deletes = 0
processed_batch_count = 0
failed_batches = []

# Process each batch
for current_batch_id, current_source in batches_to_process:
    try:
        print(f"Processing batch {current_batch_id} ({current_source})...", end=" ")
        
        # Load staging data for this batch
        staging_query = f"""
            SELECT * FROM {SILVER_SCHEMA}.silver_jobs_staging
            WHERE batch_id = '{current_batch_id}'
        """
        
        if current_source:
            staging_query += f" AND source_name = '{current_source}'"
        
        staging_df = spark.sql(staging_query)
        
        # Deduplicate staging by source_job_key (keep most recent based on processed_at)
        window_spec_staging = Window.partitionBy("source_job_key").orderBy(F.col("processed_at").desc())
        staging_df = staging_df \
            .withColumn("row_num", F.row_number().over(window_spec_staging)) \
            .filter(F.col("row_num") == 1) \
            .drop("row_num")
        
        staging_count = staging_df.count()
        
        if staging_count == 0:
            print("No records found")
            continue
        
        # Load current table
        current_df = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current")
        current_count = current_df.count()
        
        # Detect INSERTs
        inserts_df = staging_df.alias("stg").join(
            current_df.alias("cur"),
            F.col("stg.source_job_key") == F.col("cur.source_job_key"),
            "left_anti"
        ).select(
            F.expr("uuid()").alias("enterprise_job_id"),
            F.col("source_name"),
            F.col("source_job_id"),
            F.col("source_job_key"),
            F.col("company_name_raw"),
            F.col("company_name_norm"),
            F.col("title_raw"),
            F.col("title_normalized"),
            F.col("description_raw"),
            F.col("location_raw"),
            F.col("location_norm"),
            F.col("remote_type"),
            F.col("source_url"),
            F.col("posted_at"),
            F.col("last_seen"),
            F.lit(True).alias("is_active"),
            F.lit(False).alias("soft_delete_flag"),
            F.lit(None).cast("string").alias("soft_delete_reason"),
            F.col("record_hash"),
            F.col("batch_id").alias("current_batch_id"),
            run_timestamp.alias("created_at"),
            run_timestamp.alias("updated_at")
        )
        
        insert_count = inserts_df.count()
        
        # Detect UPDATEs
        updates_df = staging_df.alias("stg").join(
            current_df.alias("cur"),
            F.col("stg.source_job_key") == F.col("cur.source_job_key"),
            "inner"
        ).where(
            F.col("stg.record_hash") != F.col("cur.record_hash")
        ).select(
            F.col("cur.enterprise_job_id"),
            F.col("stg.source_name"),
            F.col("stg.source_job_id"),
            F.col("stg.source_job_key"),
            F.col("stg.company_name_raw"),
            F.col("stg.company_name_norm"),
            F.col("stg.title_raw"),
            F.col("stg.title_normalized"),
            F.col("stg.description_raw"),
            F.col("stg.location_raw"),
            F.col("stg.location_norm"),
            F.col("stg.remote_type"),
            F.col("stg.source_url"),
            F.col("stg.posted_at"),
            F.col("stg.last_seen"),
            F.lit(True).alias("is_active"),
            F.lit(False).alias("soft_delete_flag"),
            F.lit(None).cast("string").alias("soft_delete_reason"),
            F.col("stg.record_hash"),
            F.col("stg.batch_id").alias("current_batch_id"),
            F.col("cur.created_at"),
            run_timestamp.alias("updated_at")
        )
        
        update_count = updates_df.count()
        
        # Detect DELETEs - Source-aware with staleness window
        if enable_deletes and current_source:
            # Calculate staleness threshold (e.g., 7 days ago)
            staleness_threshold = F.expr(f"date_sub(current_date(), {staleness_days})")
            
            # Only soft-delete jobs that:
            # 1. Are from the SAME source as this batch
            # 2. Don't appear in this batch
            # 3. Haven't been seen in N days (stale)
            deletes_df = current_df.alias("cur").join(
                staging_df.alias("stg"),
                F.col("cur.source_job_key") == F.col("stg.source_job_key"),
                "left_anti"
            ).where(
                (F.col("cur.is_active") == True) &
                (F.col("cur.source_name") == current_source) &  # Only from same source
                (F.col("cur.last_seen") < staleness_threshold)  # Stale (not seen in N days)
            ).select(
                F.col("cur.*")
            ).withColumn("is_active", F.lit(False)) \
             .withColumn("soft_delete_flag", F.lit(True)) \
             .withColumn("soft_delete_reason", F.lit(f"Not seen in {current_source} for {staleness_days}+ days")) \
             .withColumn("updated_at", run_timestamp)
            
            delete_count = deletes_df.count()
        else:
            # Skip deletes if disabled or batch has no specific source
            deletes_df = spark.createDataFrame([], current_df.schema)
            delete_count = 0
        
        # Combine all changes
        common_columns = [
            "enterprise_job_id", "source_name", "source_job_id", "source_job_key",
            "company_name_raw", "company_name_norm", "title_raw", "title_normalized",
            "description_raw", "location_raw", "location_norm", "remote_type",
            "source_url", "posted_at", "last_seen", "is_active",
            "soft_delete_flag", "soft_delete_reason", "record_hash", "current_batch_id",
            "created_at", "updated_at"
        ]
        
        all_changes_df = inserts_df.select(*common_columns) \
            .union(updates_df.select(*common_columns)) \
            .union(deletes_df.select(*common_columns))
        
        # Deduplicate by enterprise_job_id to prevent merge conflicts
        # Priority: Use the most recent change per job (latest updated_at)
        window_spec = Window.partitionBy("enterprise_job_id").orderBy(F.col("updated_at").desc())
        all_changes_df = all_changes_df \
            .withColumn("row_num", F.row_number().over(window_spec)) \
            .filter(F.col("row_num") == 1) \
            .drop("row_num")
        
        # Add missing columns for table compatibility
        all_changes_df = all_changes_df \
            .withColumn("dq_overall_status", F.lit(None).cast("string")) \
            .withColumn("dq_validated_at", F.lit(None).cast("timestamp")) \
            .withColumn("dq_validation_level", F.lit(None).cast("string")) \
            .withColumn("sector_assigned", F.lit(None).cast("string")) \
            .withColumn("sector_assignment_method", F.lit(None).cast("string")) \
            .withColumn("sector_confidence", F.lit(None).cast("double")) \
            .withColumn("sector_assigned_at", F.lit(None).cast("timestamp"))
        
        # Merge to current table
        if current_count == 0:
            # First load - just write
            all_changes_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_SCHEMA}.silver_jobs_current")
        else:
            # Merge logic
            delta_current = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
            
            delta_current.alias("cur").merge(
                all_changes_df.alias("chg"),
                "cur.enterprise_job_id = chg.enterprise_job_id"
            ).whenMatchedUpdateAll() \
             .whenNotMatchedInsertAll() \
             .execute()
        
        # Record batch processing in metadata
        metadata_record = spark.createDataFrame([
            (current_batch_id, current_source if current_source else "all", int(insert_count), int(update_count), int(delete_count), run_timestamp_py, run_id, "success")
        ], schema=metadata_schema)
        
        metadata_record.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(metadata_table)
        
        # Update tracking
        total_inserts += insert_count
        total_updates += update_count
        total_deletes += delete_count
        processed_batch_count += 1
        
        print(f"✓ I:{insert_count} U:{update_count} D:{delete_count}")
        
    except Exception as e:
        print(f"✗ {str(e)}")
        failed_batches.append((current_batch_id, current_source, str(e)))
        
        # Record failure in metadata
        try:
            failure_record = spark.createDataFrame([
                (current_batch_id, current_source if current_source else "all", 0, 0, 0, run_timestamp_py, run_id, f"failed: {str(e)}")
            ], schema=metadata_schema)
            
            failure_record.write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(metadata_table)
        except:
            pass
        
        continue

print(f"\nProcessed {processed_batch_count} batches: {total_inserts} inserts, {total_updates} updates, {total_deletes} deletes")
if failed_batches:
    print(f"Failed: {len(failed_batches)} batches")

final_count = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current").count()

In [0]:
# Show CDC processing history
if processed_batch_count > 0:
    metadata_df = spark.sql(f"""
        SELECT batch_id, source_name, inserts, updates, deletes, processed_at, status
        FROM {metadata_table}
        ORDER BY processed_at DESC
        LIMIT 20
    """)
    display(metadata_df)
    
    # Show sample of current table
    display(spark.table(f"{SILVER_SCHEMA}.silver_jobs_current"))

In [0]:
result = {
    "status": "success" if len(failed_batches) == 0 else "partial_success",
    "run_id": run_id,
    "batches_processed": processed_batch_count,
    "batches_failed": len(failed_batches),
    "inserts": total_inserts,
    "updates": total_updates,
    "deletes": total_deletes,
    "total_changes": total_inserts + total_updates + total_deletes,
    "final_count": final_count,
    "metadata_table": metadata_table
}

dbutils.notebook.exit(json.dumps(result))